# Modelos Multimodales
### Máster en IA y Computación Cuántica Aplicada a los Mercados Financieros (MIAX)

---

- **Autor:** Pablo Hernández Cámara
- **Contacto:** [pablo.hernandez-camara@uv.es](mailto:pablo.hernandez-camara@uv.es)
- **Notebook:** Agentes, de lenguaje natural a consultas SQL

Construir agentes es muy facil usando `smolagents` de HuggingFace y los LLM que hay disponibles.

In [1]:
!pip install smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.2 MB/s eta 0:00:00


En este caso vamos a hacer un agente que nos permita hacer busquedas en tablas (con sql) usando lenguaje natural.

Lo primero vamos a crear la tabla donde el agente tendrá que buscar la información que le pidamos. En este caso usaremos SQLite en memoria. La base de datos existirá mientras el kernel mantenga viva la conexión; no se crea un archivo permanente.

La primera tabla, `receipts`, contiene un recibo por fila. Definimos `receipt_id` como clave primaria: identifica de forma única cada recibo.

In [2]:
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer, Float, insert, text, inspect
from sqlalchemy.pool import StaticPool
from smolagents import tool, CodeAgent, InferenceClientModel

engine = create_engine("sqlite:///:memory:",
                       connect_args={"check_same_thread": False},
                       poolclass=StaticPool)
metadata_obj = MetaData()

# create city SQL table
table_name = "receipts"
receipts = Table(table_name,
                 metadata_obj,
                 Column("receipt_id", Integer, primary_key=True),
                 Column("customer_name", String(16), primary_key=True),
                 Column("price", Float),
                 Column("tip", Float))

metadata_obj.create_all(engine)

Los datos son sintéticos y solo sirven para ilustrar el flujo. Insertamos todas las filas en una única transacción y después mostramos la tabla con una consulta SQL explícita.

In [3]:
rows = [{"receipt_id": 1, "customer_name": "Alan Payne", "price": 12.06, "tip": 1.20},
        {"receipt_id": 2, "customer_name": "Alex Mason", "price": 23.86, "tip": 0.24},
        {"receipt_id": 3, "customer_name": "Woodrow Wilson", "price": 53.43, "tip": 5.43},
        {"receipt_id": 4, "customer_name": "Margaret James", "price": 21.11, "tip": 1.00}]

for row in rows:
    stmt = insert(receipts).values(**row)
    with engine.begin() as connection:
        cursor = connection.execute(stmt)

In [4]:
with engine.connect() as con:
    rows = con.execute(text("""SELECT * from receipts"""))
    for row in rows:
        print(row)

(1, 'Alan Payne', 12.06, 1.2)
(2, 'Alex Mason', 23.86, 0.24)
(3, 'Woodrow Wilson', 53.43, 5.43)
(4, 'Margaret James', 21.11, 1.0)


## Construimos el agente

En este caso nuestro agente va a ser capaz de convertir lenguaje natural en sql para ejecutar las acciones que le pidamos sobre las tablas. Para ello, el agente debe tener la capacidad/herramienta para porder realizar las acciones.

Con `@tool`, `smolagents` expone una función Python que el agente puede invocar. La **docstring** es especialmente importante: ayuda al modelo a entender para qué sirve la herramienta, qué tablas existen y qué argumentos debe proporcionar.

In [5]:
@tool
def sql_engine(query: str) -> str:
    """Ejecuta una consulta SQL de solo lectura sobre la base de datos del ejemplo.

    Tablas disponibles:
    - receipts(receipt_id, customer_name, price, tip)

    Args:
        query: Consulta SQL SELECT que se desea ejecutar.

    Returns:
        Filas obtenidas, serializadas como texto, o un mensaje si no hay resultados.
    """

    normalized_query = query.strip().lower()

    # Protección mínima para el ejemplo: solo se admiten consultas que comiencen por SELECT.
    # No sustituye a un parser SQL ni a permisos de solo lectura en producción.
    if not normalized_query.startswith("select"):
        return "Consulta rechazada: esta herramienta solo permite consultas SELECT."

    try:
        with engine.connect() as connection:
            result = connection.execute(text(query))
            rows = result.fetchall()
            if not rows:
                return "La consulta no ha devuelto filas."
            return "\n".join(str(tuple(row)) for row in rows)
    except Exception as exc:
        # Devolvemos el error para facilitar la depuración en este notebook docente.
        return f"Error al ejecutar SQL: {type(exc).__name__}: {exc}"

El modelo propone los pasos necesarios y decide cuándo llamar a `sql_engine`. La herramienta realiza la consulta real; el modelo no obtiene los datos por conocimiento propio.

Hay que estar autenticarte en Hugging Face y disponer de acceso al modelo seleccionado.

In [6]:
from huggingface_hub import notebook_login

notebook_login()

In [7]:
model = InferenceClientModel(model_id="meta-llama/Llama-3.1-8B-Instruct")

agent = CodeAgent(tools=[sql_engine],
                  model=model)

Vamos a probarlo: Imaginad que queremos encontrar que cliente tiene el mayor recibo.

Pregunta en lenguaje natural: **¿Qué cliente tiene el recibo de mayor importe?**

Una consulta SQL posible es:

```sql
SELECT customer_name, price
FROM receipts
ORDER BY price DESC
LIMIT 1;
```

Al agente le damos la pregunta en lenguaje natural. Después observamos si el agente selecciona la herramienta y genera una consulta adecuada. El resultado puede variar según el modelo, sus instrucciones y la versión de la biblioteca.

In [8]:
question = ("Which customer has the most expensive receipt? Use the sql_engine tool to look up the answer.")

agent.run(question)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which customer has the most expensive receipt? Use the sql_engine tool to look up the answer.                   │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = "SELECT customer_name FROM receipts ORDER BY price DESC LIMIT 1"                                         
  result = sql_engine(query)                                                                                       
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
('Woodrow Wilson',)

Out: None

[Step 1: Duration 5.46 seconds| Input tokens: 2,072 | Output tokens: 107]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: ('Woodrow Wilson',)

[Step 3: Duration 2.33 seconds| Input tokens: 6,788 | Output tokens: 238]

"('Woodrow Wilson',)"

Ha tenido algún error al escribir la consulta de sql pero lo ha conseguido hacer. Mejores modelos tendrán menos errores.

Vamos a complicarselo aun más, a ver que es capaz de plantear. Ahora incorporamos `waiters`, que relaciona cada recibo con el camarero correspondiente. Para responder va a tener que juntar la información de ambas tablas (join).

In [9]:
table_name = "waiters"
receipts = Table(table_name, metadata_obj, Column("receipt_id", Integer, primary_key=True), Column("waiter_name", String(16), primary_key=True),)
metadata_obj.create_all(engine)

rows = [{"receipt_id": 1, "waiter_name": "Corey Johnson"},
        {"receipt_id": 2, "waiter_name": "Michael Watts"},
        {"receipt_id": 3, "waiter_name": "Michael Watts"},
        {"receipt_id": 4, "waiter_name": "Margaret James"}]

for row in rows:
    stmt = insert(receipts).values(**row)
    with engine.begin() as connection:
        cursor = connection.execute(stmt)

In [10]:
with engine.connect() as con:
    rows = con.execute(text("""SELECT * from receipts"""))
    for row in rows:
        print(row)

(1, 'Alan Payne', 12.06, 1.2)
(2, 'Alex Mason', 23.86, 0.24)
(3, 'Woodrow Wilson', 53.43, 5.43)
(4, 'Margaret James', 21.11, 1.0)


In [11]:
with engine.connect() as con:
    rows = con.execute(text("""SELECT * from waiters"""))
    for row in rows:
        print(row)

(1, 'Corey Johnson')
(2, 'Michael Watts')
(3, 'Michael Watts')
(4, 'Margaret James')


Las herramientas no solo necesitan código: también necesitan una descripción útil para el modelo. En este caso, generamos la descripción de las tablas a partir del esquema real de la base de datos.

Además de las columnas, es importante explicar la relación entre las tablas: `receipts.receipt_id = waiters.receipt_id`. En general, cuanta más información le demos al modelo mejor, por lo que el esquema ayudara al modelo a escribir SQL, pero no elimina la posibilidad de errores.

In [12]:
#updated_description = """Allows you to perform SQL queries on the table. Beware that this tool's output is a string representation of the execution output.
#It can use the following tables:"""
#
#inspector = inspect(engine)
#for table in ["receipts", "waiters"]:
#    columns_info = [(col["name"], col["type"]) for col in inspector.get_columns(table)]
#
#    table_description = f"Table '{table}':\n"
#
#    table_description += "Columns:\n" + "\n".join([f"  - {name}: {col_type}" for name, col_type in columns_info])
#    updated_description += "\n\n" + table_description
#
#print(updated_description)

inspector = inspect(engine)

table_descriptions = [
    "Ejecuta consultas SQL SELECT de solo lectura sobre estas tablas:"
]

for table_name in ("receipts", "waiters"):
    columns = inspector.get_columns(table_name)
    column_lines = [
        f"  - {column['name']}: {column['type']}"
        for column in columns
    ]
    table_descriptions.append(
        f"\nTabla '{table_name}':\n" + "\n".join(column_lines)
    )

table_descriptions.append(
    "\nRelación: receipts.receipt_id = waiters.receipt_id."
)
updated_description = "\n".join(table_descriptions)

print(updated_description)

Ejecuta consultas SQL SELECT de solo lectura sobre estas tablas:

Tabla 'receipts':
  - receipt_id: INTEGER
  - customer_name: VARCHAR(16)
  - price: FLOAT
  - tip: FLOAT

Tabla 'waiters':
  - receipt_id: INTEGER
  - waiter_name: VARCHAR(16)

Relación: receipts.receipt_id = waiters.receipt_id.


Vamos a hacer un nuevo agente con un modelo mejor, i.e. más grande en este caso. Podriamos probar con el mismo de antes, pero esta tarea es más complicada.

Vamos a preguntarle por que camarero ha recibido más propinas. Esta consulta requiere combinar las tablas y sumar `tip` por camarero. Una consulta SQL de referencia para comprobar el resultado sería:

```sql
SELECT w.waiter_name, SUM(r.tip) AS total_tips
FROM waiters AS w
JOIN receipts AS r ON r.receipt_id = w.receipt_id
GROUP BY w.waiter_name
ORDER BY total_tips DESC
LIMIT 1;
```



In [13]:
sql_engine.description = updated_description

agent = CodeAgent(tools=[sql_engine],
                  model=InferenceClientModel("Qwen/Qwen2.5-72B-Instruct"))

answer = agent.run("Which waiter received the greatest total amount in tips?. Use the sql_engine tool and account for all receipts.")
answer

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which waiter received the greatest total amount in tips?. Use the sql_engine tool and account for all receipts. │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = """                                                                                                      
  SELECT waiter_name, SUM(tip) AS total_tips                                                                       
  FROM receipts                                                                                                    
  JOIN waiters ON receipts.receipt_id = waiters.receipt_id                                                         
  GROUP BY waiter_name                                                                                             
  ORDER BY total_tips DESC                                                                                         
  LIMIT 1;                                                                                                         
  """                                                                                                              
  result = sql_engine(query)                                                                                       
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
('Michael Watts', 5.67)

Out: None

[Step 1: Duration 6.63 seconds| Input tokens: 2,166 | Output tokens: 129]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Michael Watts")                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Michael Watts

[Step 2: Duration 2.04 seconds| Input tokens: 4,598 | Output tokens: 166]

'Michael Watts'

Ha respondido bien? Vamos a comprobarlo:

In [14]:
reference_query = text("""
    SELECT w.waiter_name, SUM(r.tip) AS total_tips
    FROM waiters AS w
    JOIN receipts AS r ON r.receipt_id = w.receipt_id
    GROUP BY w.waiter_name
    ORDER BY total_tips DESC
    LIMIT 1
""")

with engine.connect() as connection:
    reference_result = connection.execute(reference_query).all()

print("Resultado de referencia:", reference_result)
print("Respuesta del agente:", answer)

Resultado de referencia: [('Michael Watts', 5.67)]
Respuesta del agente: Michael Watts
